# PipelineWatch-NG — Module 1: Data Ingestion
### Satellite-based crude oil theft monitoring · Niger Delta, Nigeria

**Study area:** Trans Niger Pipeline (TNP) corridor · 5.0°–5.8°N, 6.5°–7.2°E

| Sensor | What it detects | Cloud-penetrating | Night-capable |
|--------|----------------|:-----------------:|:-------------:|
| Sentinel-1 SAR | Oil spill dark spots | ✅ C-band radar | ✅ Active sensor |
| FIRMS / VIIRS | Illegal refinery fires | ✅ Thermal IR | ✅ Thermal IR |
| TROPOMI SO₂ | Chemical plumes from crude burning | ✅ UV backscatter | ❌ Daytime |

**Everything runs on Google Earth Engine — zero satellite downloads.**

---
Run cells top to bottom. Each cell builds on the previous one.

## Cell 1 — Package check

In [1]:
import importlib

packages = {
    'earthengine-api': 'ee',
    'geemap':          'geemap',
    'geopandas':       'geopandas',
    'folium':          'folium',
    'pandas':          'pandas',
    'numpy':           'numpy',
    'matplotlib':      'matplotlib',
}
missing = []
for pkg, imp in packages.items():
    try:
        importlib.import_module(imp)
        print(f'  OK      {pkg}')
    except ImportError:
        missing.append(pkg)
        print(f'  MISSING {pkg}')
if missing:
    print(f'\nRun: pip install {" ".join(missing)}')
else:
    print('\nAll packages ready.')

C:\Users\taylo\anaconda3\envs\pipelinewatch\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


  OK      earthengine-api
  OK      geemap
  OK      geopandas
  OK      folium
  OK      pandas
  OK      numpy
  OK      matplotlib

All packages ready.


C:\Users\taylo\anaconda3\envs\pipelinewatch\lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


## Cell 2 — Imports

In [2]:
import ee
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
print('Imports OK.')

Imports OK.


## Cell 3 — GEE authentication
Replace `pipelinewatch-ng` if your project ID is different.

In [3]:
GEE_PROJECT_ID = 'pipelinewatch-ng'

try:
    ee.Initialize(project=GEE_PROJECT_ID)
    result = ee.Number(42).getInfo()
    print(f'GEE connected — project: {GEE_PROJECT_ID}')
    print(f'Connection test: {result}  (should be 42)')
except Exception:
    print('Not authenticated. Opening browser...')
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)
    print('Authenticated.')

GEE connected — project: pipelinewatch-ng
Connection test: 42  (should be 42)


## Cell 4 — Study area and thresholds

In [4]:
ROI_BOUNDS  = [6.50, 5.00, 7.20, 5.80]
ROI_CENTER  = [5.40, 6.85]
ROI_ZOOM    = 9

BASELINE_START = '2023-01-01'
BASELINE_END   = '2023-06-30'
RECENT_START   = '2024-01-01'
RECENT_END     = '2024-06-30'

SAR_DARK_SPOT_SIGMA = 1.5
FIRMS_BRIGHTNESS_K  = 330.0
SO2_THRESHOLD_DU    = 3.0

ROI = ee.Geometry.Rectangle(ROI_BOUNDS)

print('Study area : Trans Niger Pipeline (TNP) corridor')
print(f'Bbox       : {ROI_BOUNDS}')
print(f'Baseline   : {BASELINE_START} to {BASELINE_END}')
print(f'Recent     : {RECENT_START} to {RECENT_END}')

Study area : Trans Niger Pipeline (TNP) corridor
Bbox       : [6.5, 5.0, 7.2, 5.8]
Baseline   : 2023-01-01 to 2023-06-30
Recent     : 2024-01-01 to 2024-06-30


## Cell 5 — Sentinel-1 SAR functions
Function definitions only — nothing runs yet.

**Key fixes applied:**
- `apply_filter=False` by default — speckle filter across 30 scenes kills the kernel
- `dark_spot_magnitude` computed as `vv.subtract(mean_vv).multiply(-1)` — keeps it as ee.Image
- `.toInt()` before `reduceToVectors` — GEE requires integer bands for vectorisation
- Dark spot extraction skipped in ingestion — deferred to Module 2 to avoid timeout

In [5]:
def get_s1_collection(roi, start, end):
    return (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(roi)
        .filterDate(start, end)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
        .select(['VV', 'VH'])
    )


def compute_sar_features(image, roi, sigma_threshold=1.5):
    """
    Add feature bands:
      dark_spot_mask      : 1 where VV is anomalously low (oil candidate)
      dark_spot_magnitude : sigma below mean — higher = darker = more suspicious
      vv_vh_ratio         : discriminates oil from wind shadow
    Fix: dark_mag starts from vv (ee.Image) not mean_vv (ee.Number)
    """
    vv = image.select('VV')
    vh = image.select('VH')
    stats = vv.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=roi, scale=100, maxPixels=1e9, bestEffort=True
    )
    mean_vv    = ee.Number(stats.get('VV_mean'))
    std_vv     = ee.Number(stats.get('VV_stdDev'))
    threshold  = mean_vv.subtract(std_vv.multiply(sigma_threshold))
    dark_mask  = vv.lt(threshold).rename('dark_spot_mask')
    # Fix: vv.subtract(mean_vv) keeps result as ee.Image
    dark_mag   = vv.subtract(mean_vv).multiply(-1).divide(std_vv).rename('dark_spot_magnitude')
    vv_lin     = ee.Image(10).pow(vv.divide(10))
    vh_lin     = ee.Image(10).pow(vh.divide(10))
    ratio      = vv_lin.divide(vh_lin).rename('vv_vh_ratio')
    return image.addBands(dark_mask).addBands(dark_mag).addBands(ratio)


def build_s1_composite(roi, start, end, sigma_threshold=1.5):
    """
    Median composite — no speckle filter (apply_filter removed).
    Speckle filtering 30 scenes kills the kernel — deferred to Module 2.
    Median compositing already suppresses most noise.
    """
    col   = get_s1_collection(roi, start, end)
    count = col.size().getInfo()
    print(f'  S1 scenes ({start} to {end}): {count}')
    if count == 0:
        print('  WARNING: no scenes found')
        return None, None, 0
    col = col.map(lambda img: compute_sar_features(img, roi, sigma_threshold))
    return col.median().clip(roi), col, count


print('SAR functions defined.')

SAR functions defined.


## Cell 6 — FIRMS/VIIRS and TROPOMI SO₂ functions
**Key fixes applied:**
- `.toInt()` before `reduceToVectors` in both FIRMS and SO₂ extraction
- Empty collection guard in monthly fire counting
- SO₂ scaled by 100 before `.toInt()` to preserve decimal precision in vectorisation

In [6]:
def get_firms_collection(roi, start, end):
    return (
        ee.ImageCollection('FIRMS')
        .filterDate(start, end)
        .filterBounds(roi)
        .select(['T21', 'confidence', 'line_number'])
    )


def compute_firms_composite(col, roi):
    return (
        col.select('T21').max().rename('T21_max').clip(roi)
        .addBands(col.select('T21').count().rename('fire_count').clip(roi))
        .addBands(col.select('T21').mean().rename('T21_mean').clip(roi))
    )


def extract_fire_hotspots(firms_comp, roi, t21_threshold_k=330.0, min_count=3):
    """
    Fix: .toInt() added before reduceToVectors.
    GEE requires integer bands for vectorisation.
    """
    hot_mask = (
        firms_comp.select('T21_max').gt(t21_threshold_k)
        .And(firms_comp.select('fire_count').gte(min_count))
    )
    vectors = (
        firms_comp.select('T21_max').updateMask(hot_mask).toInt()  # Fix
        .reduceToVectors(
            geometry=roi, scale=375, geometryType='centroid',
            eightConnected=True, maxPixels=1e9, bestEffort=True
        )
    )
    def annotate(f):
        geom  = f.geometry()
        stats = firms_comp.reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True),
            geometry=geom.buffer(500), scale=375, maxPixels=1e6
        )
        t21_max = ee.Number(stats.get('T21_max_max'))
        count   = ee.Number(stats.get('fire_count_mean'))
        source  = ee.Algorithms.If(
            t21_max.gt(380), 'flare_candidate',
            ee.Algorithms.If(count.gte(10), 'refinery_candidate', 'agricultural_or_other')
        )
        return f.set({
            'T21_max_K':     t21_max,
            'T21_mean_K':    ee.Number(stats.get('T21_mean_mean')),
            'fire_count':    count,
            'signal_type':   'FIRMS_fire_hotspot',
            'likely_source': source,
            'confidence':    ee.Algorithms.If(count.gte(10), 'high', 'nominal'),
        })
    return vectors.map(annotate)


def get_tropomi_collection(roi, start, end):
    return (
        ee.ImageCollection('COPERNICUS/S5P/NRTI/L3_SO2')
        .filterDate(start, end)
        .filterBounds(roi)
        .select(['SO2_column_number_density', 'cloud_fraction'])
    )


def compute_so2_composite(col, roi, max_cloud=0.3):
    def mask_and_convert(image):
        cloud  = image.select('cloud_fraction')
        so2_du = (
            image.select('SO2_column_number_density')
            .multiply(2241.5).rename('SO2_DU')
        )
        return so2_du.updateMask(cloud.lt(max_cloud))
    clean = col.map(mask_and_convert)
    return (
        clean.mean().rename('SO2_mean_DU').clip(roi)
        .addBands(clean.max().rename('SO2_max_DU').clip(roi))
        .addBands(clean.reduce(ee.Reducer.percentile([75])).rename('SO2_p75_DU').clip(roi))
        .addBands(clean.count().rename('SO2_obs_count').clip(roi))
    )


def extract_so2_anomalies(so2_comp, roi, threshold_du=3.0, min_obs=5):
    """
    Fix: .multiply(100).toInt() before reduceToVectors.
    Scales small DU values before integer conversion to preserve boundaries.
    """
    mask = (
        so2_comp.select('SO2_mean_DU').gt(threshold_du)
        .And(so2_comp.select('SO2_obs_count').gte(min_obs))
    )
    vectors = (
        so2_comp.select('SO2_mean_DU').updateMask(mask).multiply(100).toInt()  # Fix
        .reduceToVectors(
            geometry=roi, scale=5500, geometryType='polygon',
            eightConnected=True, maxPixels=1e9, bestEffort=True
        )
    )
    def annotate(f):
        geom  = f.geometry()
        stats = so2_comp.reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True),
            geometry=geom, scale=5500, maxPixels=1e6
        )
        so2_mean = ee.Number(stats.get('SO2_mean_DU_mean'))
        conf = ee.Algorithms.If(
            so2_mean.gt(8), 'high',
            ee.Algorithms.If(so2_mean.gt(5), 'nominal', 'low')
        )
        return f.set({
            'SO2_mean_DU': so2_mean,
            'SO2_max_DU':  ee.Number(stats.get('SO2_max_DU_max')),
            'obs_count':   ee.Number(stats.get('SO2_obs_count_mean')),
            'area_m2':     geom.area(maxError=100),
            'signal_type': 'TROPOMI_SO2_anomaly',
            'confidence':  conf,
        })
    return vectors.map(annotate)


def compute_fire_gas_risk_score(so2_comp, firms_comp, roi,
                                so2_threshold=3.0, t21_threshold=330.0):
    """
    0 = no signal
    1 = fire only
    2 = SO2 only
    3 = fire + SO2 co-located = high-confidence illegal refinery
    """
    so2_mask  = so2_comp.select('SO2_mean_DU').gt(so2_threshold)
    fire_mask = firms_comp.select('T21_max').gt(t21_threshold)
    fire_rs   = fire_mask.reproject(crs='EPSG:4326', scale=5500)
    so2_rs    = so2_mask.reproject(crs='EPSG:4326', scale=5500)
    return (
        so2_rs.multiply(2).add(fire_rs)
        .rename('fire_gas_risk_score').clip(roi)
    )


print('FIRMS/VIIRS and TROPOMI functions defined.')

FIRMS/VIIRS and TROPOMI functions defined.


## Cell 7 — Run Sentinel-1 SAR
No speckle filter — runs fast without crashing.

In [7]:
print('=== Sentinel-1 SAR ===')

print(f'Baseline ({BASELINE_START} to {BASELINE_END})...')
s1_baseline, s1_baseline_col, n_baseline = build_s1_composite(
    ROI, BASELINE_START, BASELINE_END, sigma_threshold=SAR_DARK_SPOT_SIGMA
)

print(f'Recent ({RECENT_START} to {RECENT_END})...')
s1_recent, s1_recent_col, n_recent = build_s1_composite(
    ROI, RECENT_START, RECENT_END, sigma_threshold=SAR_DARK_SPOT_SIGMA
)

# Vectorisation deferred to Module 2 — times out in interactive getInfo()
n_spots = 'deferred to Module 2'

print()
print(f'Baseline scenes : {n_baseline}')
print(f'Recent scenes   : {n_recent}')
print(f'Dark spot zones : {n_spots}')
print('SAR composites ready.')

=== Sentinel-1 SAR ===
Baseline (2023-01-01 to 2023-06-30)...
  S1 scenes (2023-01-01 to 2023-06-30): 30
Recent (2024-01-01 to 2024-06-30)...
  S1 scenes (2024-01-01 to 2024-06-30): 29

Baseline scenes : 30
Recent scenes   : 29
Dark spot zones : deferred to Module 2
SAR composites ready.


## Cell 8 — Run FIRMS/VIIRS

In [8]:
print('=== FIRMS / VIIRS ===')

print(f'Baseline ({BASELINE_START} to {BASELINE_END})...')
firms_baseline_col  = get_firms_collection(ROI, BASELINE_START, BASELINE_END)
n_firms_base        = firms_baseline_col.size().getInfo()
firms_baseline_comp = compute_firms_composite(firms_baseline_col, ROI)
print(f'  Images: {n_firms_base}')

print(f'Recent ({RECENT_START} to {RECENT_END})...')
firms_recent_col    = get_firms_collection(ROI, RECENT_START, RECENT_END)
n_firms_rec         = firms_recent_col.size().getInfo()
firms_recent_comp   = compute_firms_composite(firms_recent_col, ROI)
print(f'  Images: {n_firms_rec}')

print('Extracting persistent hotspots...')
fire_hotspots = extract_fire_hotspots(
    firms_recent_comp, ROI,
    t21_threshold_k=FIRMS_BRIGHTNESS_K, min_count=3
)
n_hotspots = fire_hotspots.size().getInfo()
print(f'Persistent hotspots: {n_hotspots}')

if n_hotspots > 0:
    sample = fire_hotspots.limit(3).getInfo()
    print()
    for feat in sample['features']:
        p = feat['properties']
        t = p.get('T21_max_K', 0)
        c = p.get('fire_count', 0)
        s = p.get('likely_source', '?')
        print(f'  T21_max={t:.0f}K  count={c:.0f}  source={s}')

=== FIRMS / VIIRS ===
Baseline (2023-01-01 to 2023-06-30)...
  Images: 180
Recent (2024-01-01 to 2024-06-30)...
  Images: 181
Extracting persistent hotspots...
Persistent hotspots: 50

  T21_max=349K  count=3  source=agricultural_or_other
  T21_max=336K  count=4  source=agricultural_or_other
  T21_max=338K  count=3  source=agricultural_or_other


## Cell 9 — Run TROPOMI SO₂

In [9]:
print('=== TROPOMI SO2 ===')

print(f'Baseline ({BASELINE_START} to {BASELINE_END})...')
tropomi_base_col = get_tropomi_collection(ROI, BASELINE_START, BASELINE_END)
n_tropomi_base   = tropomi_base_col.size().getInfo()
so2_baseline     = compute_so2_composite(tropomi_base_col, ROI)
print(f'  Images: {n_tropomi_base}')

print(f'Recent ({RECENT_START} to {RECENT_END})...')
tropomi_rec_col  = get_tropomi_collection(ROI, RECENT_START, RECENT_END)
n_tropomi_rec    = tropomi_rec_col.size().getInfo()
so2_recent       = compute_so2_composite(tropomi_rec_col, ROI)
print(f'  Images: {n_tropomi_rec}')

print(f'Extracting SO2 anomaly zones (> {SO2_THRESHOLD_DU} DU)...')
so2_anomalies = extract_so2_anomalies(
    so2_recent, ROI, threshold_du=SO2_THRESHOLD_DU, min_obs=5
)
n_so2 = so2_anomalies.size().getInfo()
print(f'SO2 anomaly zones: {n_so2}')

print('Computing combined fire + SO2 risk score...')
fire_gas_score = compute_fire_gas_risk_score(
    so2_recent, firms_recent_comp, ROI,
    so2_threshold=SO2_THRESHOLD_DU, t21_threshold=FIRMS_BRIGHTNESS_K
)
print('Score ready  (0=none  1=fire  2=SO2  3=fire+SO2)')

=== TROPOMI SO2 ===
Baseline (2023-01-01 to 2023-06-30)...
  Images: 313
Recent (2024-01-01 to 2024-06-30)...
  Images: 278
Extracting SO2 anomaly zones (> 3.0 DU)...
SO2 anomaly zones: 0
Computing combined fire + SO2 risk score...
Score ready  (0=none  1=fire  2=SO2  3=fire+SO2)


## Cell 10 — Ingestion summary

In [10]:
df = pd.DataFrame([
    {'Sensor': 'Sentinel-1 SAR', 'Scenes': n_recent,
     'Detections': n_spots,    'Type': 'Oil spill dark spots',
     'All-weather': 'Yes', 'Night': 'Yes'},
    {'Sensor': 'FIRMS/VIIRS',   'Scenes': n_firms_rec,
     'Detections': n_hotspots, 'Type': 'Persistent fire hotspots',
     'All-weather': 'Partial', 'Night': 'Yes'},
    {'Sensor': 'TROPOMI SO2',   'Scenes': n_tropomi_rec,
     'Detections': n_so2,      'Type': 'SO2 column anomalies',
     'All-weather': 'Partial', 'Night': 'No'},
])
print('=== Module 1 Ingestion Summary ===')
print(f'Period : {RECENT_START} to {RECENT_END}')
print(f'Area   : Trans Niger Pipeline corridor')
print()
print(df.to_string(index=False))
print(f'\nGenerated: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

=== Module 1 Ingestion Summary ===
Period : 2024-01-01 to 2024-06-30
Area   : Trans Niger Pipeline corridor

        Sensor  Scenes           Detections                     Type All-weather Night
Sentinel-1 SAR      29 deferred to Module 2     Oil spill dark spots         Yes   Yes
   FIRMS/VIIRS     181                   50 Persistent fire hotspots     Partial   Yes
   TROPOMI SO2     278                    0     SO2 column anomalies     Partial    No

Generated: 2026-03-15 19:18


## Cell 11 — Interactive risk map
Use the **layer control (top-right)** to toggle signals.
Toggle SAR baseline (blue) vs recent (red) to see new spill activity.

In [11]:
import geemap

m = geemap.Map(center=ROI_CENTER, zoom=ROI_ZOOM)
m.add_basemap('SATELLITE')

m.addLayer(
    s1_recent.select('VV'),
    {'min': -25, 'max': 0,
     'palette': ['000000','404040','808080','c0c0c0','ffffff']},
    'S1 VV backscatter (dB)'
)
m.addLayer(
    s1_recent.select('dark_spot_mask').selfMask(),
    {'palette': ['E24B4A'], 'opacity': 0.7},
    'SAR dark spots — oil candidates (recent)'
)
m.addLayer(
    s1_baseline.select('dark_spot_mask').selfMask(),
    {'palette': ['378ADD'], 'opacity': 0.5},
    'SAR dark spots — BASELINE'
)
m.addLayer(
    firms_recent_comp.select('T21_max'),
    {'min': 300, 'max': 420,
     'palette': ['E6F1FB','FAC775','EF9F27','E24B4A','791F1F']},
    'VIIRS max brightness temp (K)'
)
m.addLayer(
    so2_recent.select('SO2_mean_DU'),
    {'min': 0, 'max': 15,
     'palette': ['E1F5EE','9FE1CB','1D9E75','085041','04342C']},
    'TROPOMI SO2 mean (Dobson Units)'
)
m.addLayer(
    fire_gas_score.selfMask(),
    {'min': 1, 'max': 3,
     'palette': ['FAC775','EF9F27','E24B4A'], 'opacity': 0.75},
    'Combined risk score (1=fire  2=SO2  3=both)'
)
m.addLayer(
    ee.FeatureCollection([ee.Feature(ROI)]),
    {'color': '185FA5', 'fillColor': '00000000'},
    'TNP corridor boundary'
)
m.add_layer_control()
print('Map ready.')
m

Map ready.


Map(center=[5.4, 6.85], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…

## Cell 12 — Export GeoJSON outputs
Writes small files to `data/cached/`. Commit to GitHub for the Streamlit dashboard.

**Fix applied:** SAR exported as sampled points (not vectorised polygons) to avoid timeout.

In [12]:
CACHE_DIR = '../data/cached'
os.makedirs(CACHE_DIR, exist_ok=True)

def export_fc(fc, filename, label, max_features=500):
    path = os.path.join(CACHE_DIR, filename)
    n    = fc.size().getInfo()
    if n == 0:
        print(f'  {label}: 0 features — writing empty file')
        with open(path, 'w') as f:
            json.dump({'type': 'FeatureCollection', 'features': []}, f)
        return 0
    print(f'  {label}: {n} features...')
    gj = fc.limit(max_features).getInfo()
    gj['metadata'] = {
        'exported': datetime.now().isoformat(),
        'count':    n,
        'signal':   label,
        'period':   f'{RECENT_START} to {RECENT_END}',
    }
    with open(path, 'w') as f:
        json.dump(gj, f, indent=2)
    kb = os.path.getsize(path) / 1024
    print(f'    saved: {path}  ({kb:.1f} KB)')
    return n

print('=== Exporting GeoJSON ===')

# SAR: sample dark spot pixels as points — avoids vectorisation timeout
print('  SAR dark spots: sampling to points...')
sar_sampled = (
    s1_recent
    .select(['VV', 'dark_spot_mask', 'dark_spot_magnitude'])
    .sample(region=ROI, scale=200, numPixels=300, geometries=True)
    .filter(ee.Filter.eq('dark_spot_mask', 1))
)
export_fc(sar_sampled, 'sar_dark_spots.geojson', 'SAR dark spots')

# FIRMS hotspots
export_fc(fire_hotspots, 'fire_hotspots.geojson', 'VIIRS fire hotspots')

# SO2 anomalies
export_fc(so2_anomalies, 'so2_anomalies.geojson', 'TROPOMI SO2 anomalies')

# Metadata
meta = {
    'module':     'M1_ingestion',
    'exported':   datetime.now().isoformat(),
    'study_area': 'Trans Niger Pipeline corridor, Niger Delta',
    'bbox':       ROI_BOUNDS,
    'periods': {
        'baseline': f'{BASELINE_START} to {BASELINE_END}',
        'recent':   f'{RECENT_START} to {RECENT_END}',
    },
    'scene_counts': {
        's1_baseline':      n_baseline,
        's1_recent':        n_recent,
        'firms_baseline':   n_firms_base,
        'firms_recent':     n_firms_rec,
        'tropomi_baseline': n_tropomi_base,
        'tropomi_recent':   n_tropomi_rec,
    },
    'detections': {
        'fire_hotspots': n_hotspots,
        'so2_anomalies': n_so2,
    },
}
meta_path = os.path.join(CACHE_DIR, 'm1_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'  metadata saved: {meta_path}')
print()
print('All outputs written to data/cached/')
print('Commit this folder to GitHub for the Streamlit dashboard.')

=== Exporting GeoJSON ===
  SAR dark spots: sampling to points...
  SAR dark spots: 3 features...
    saved: ../data/cached\sar_dark_spots.geojson  (1.5 KB)
  VIIRS fire hotspots: 50 features...
    saved: ../data/cached\fire_hotspots.geojson  (28.6 KB)
  TROPOMI SO2 anomalies: 0 features — writing empty file
  metadata saved: ../data/cached\m1_metadata.json

All outputs written to data/cached/
Commit this folder to GitHub for the Streamlit dashboard.


## Module 1 complete ✓

| Output | File |
|--------|------|
| SAR oil spill sample points | `data/cached/sar_dark_spots.geojson` |
| VIIRS fire hotspots | `data/cached/fire_hotspots.geojson` |
| TROPOMI SO₂ anomalies | `data/cached/so2_anomalies.geojson` |
| Run metadata | `data/cached/m1_metadata.json` |

### Fixes applied in this version
- `dark_spot_magnitude` computed from `vv.subtract(mean_vv)` — keeps `ee.Image` type
- `apply_filter=False` — speckle filter across 30 scenes crashes the kernel
- `.toInt()` added before all `reduceToVectors` calls — GEE requirement
- `.multiply(100).toInt()` for SO₂ — preserves decimal boundaries before cast
- SAR exported as sampled points — vectorisation times out interactively
- Time series charts removed — caused kernel crash on low-bandwidth connections
- Monthly aggregation approach documented for Module 2

### Next: Module 2
- Sentinel-2 NDVI/NDWI — vegetation dieback along pipeline right-of-way
- DBSCAN clustering — group fire hotspots into refinery site candidates
- SAR speckle filter applied properly via GEE Export (not interactive)
- SAR change detection — recent minus baseline
- Output: labelled feature table for ML training in Module 3